# LeetCode 401 — Binary Watch bằng Python

**Link đề:** https://leetcode.com/problems/binary-watch/
**Độ khó:** Easy · **Chủ đề:** Backtracking, Bit Manipulation

```python
class Solution:
    def readBinaryWatch(self, turnedOn: int) -> list[str]:
        ...
```

## Mục tiêu học
1. Hiểu cách đồng hồ nhị phân biểu diễn giờ/phút bằng **bit** và đếm số **bit 1** (`popcount`).
2. Đổi góc nhìn: thay vì "chọn đèn nào sáng" (khó), hãy **liệt kê mọi thời điểm hợp lệ** (chỉ 12 × 60 = 720) rồi lọc — một ví dụ đẹp của "**không gian đáp án nhỏ thì vét cạn đáp án**".
3. Nắm `bin(x).count('1')`, `int.bit_count()`, phép `&`, `>>`, `<<` và định dạng chuỗi `f"{m:02d}"`.
4. Biết thêm lời giải **quay lui (backtracking)** chọn đèn và lời giải **duyệt mặt nạ bit (bitmask)** 10 bit.
5. Trace, kiểm thử đối chiếu, và các lỗi định dạng hay gặp (`"10:2"`, `"01:00"`).

## 1. Đề bài

Đồng hồ nhị phân có **4 đèn LED ở trên** biểu diễn **giờ (0–11)** và **6 đèn ở dưới** biểu diễn **phút (0–59)**. Mỗi đèn là một bit 0/1, **bit có trọng số thấp nhất ở bên phải**.

- 4 đèn giờ có giá trị `8, 4, 2, 1`.
- 6 đèn phút có giá trị `32, 16, 8, 4, 2, 1`.

Ví dụ: đèn giờ `0100` (= 4) và đèn phút `110011` (= 32 + 16 + 2 + 1 = 51) → đồng hồ chỉ `"4:51"`.

Cho số nguyên `turnedOn` = **số đèn đang sáng** (bỏ qua AM/PM). Trả về **mọi thời điểm** đồng hồ có thể đang hiển thị, thứ tự tùy ý.

**Định dạng:**
- Giờ **không** có số 0 ở đầu: `"01:00"` sai, phải là `"1:00"`.
- Phút **luôn 2 chữ số**: `"10:2"` sai, phải là `"10:02"`.

| `turnedOn` | Output |
|---:|---|
| 1 | `["0:01","0:02","0:04","0:08","0:16","0:32","1:00","2:00","4:00","8:00"]` |
| 9 | `[]` |

**Constraints:** `0 <= turnedOn <= 10`.

In [1]:
# Kiểm tra ví dụ "4:51" bằng code
hour_leds = "0100"
minute_leds = "110011"
print("giờ =", int(hour_leds, 2), "| phút =", int(minute_leds, 2))
print("số đèn sáng =", hour_leds.count("1") + minute_leds.count("1"))

giờ = 4 | phút = 51
số đèn sáng = 5


`int(chuỗi, 2)` đọc chuỗi như số **nhị phân** (cơ số 2): `int("0100", 2) = 4`.

**Vì sao `turnedOn = 9` cho `[]`?** Giờ tối đa 11 = `1011` (3 bit 1) và phút tối đa 59 = `111011` (5 bit 1). Không thể bật cả 4 đèn giờ (= 15 > 11) hay cả 6 đèn phút (= 63 > 59). Tối đa chỉ `3 + 5 = 8` đèn → 9, 10 đèn đều không có thời điểm hợp lệ.

## 2. Nền tảng Python

### 2.1. `bin(x)` và đếm bit 1

`bin(x)` trả chuỗi nhị phân có tiền tố `'0b'`. `bin(x).count('1')` đếm số bit 1. Từ Python 3.10 có `x.bit_count()` làm việc này nhanh hơn.

In [2]:
for x in [0, 1, 4, 11, 51, 59]:
    print(f"{x:>2} -> bin={bin(x):>9} -> số bit 1 = {bin(x).count('1')} | bit_count() = {x.bit_count()}")

 0 -> bin=      0b0 -> số bit 1 = 0 | bit_count() = 0
 1 -> bin=      0b1 -> số bit 1 = 1 | bit_count() = 1
 4 -> bin=    0b100 -> số bit 1 = 1 | bit_count() = 1
11 -> bin=   0b1011 -> số bit 1 = 3 | bit_count() = 3
51 -> bin= 0b110011 -> số bit 1 = 4 | bit_count() = 4
59 -> bin= 0b111011 -> số bit 1 = 5 | bit_count() = 5


### 2.2. Định dạng chuỗi với f-string

- `f"{h}"` in số bình thường (không có 0 đầu).
- `f"{m:02d}"`: độ rộng **2**, đệm **0** bên trái, kiểu số nguyên `d`. Ví dụ `2 → "02"`, `32 → "32"`.

In [3]:
for h, m in [(0, 1), (10, 2), (4, 51), (11, 59)]:
    print(f"{h}:{m:02d}", "| sai nếu không đệm:", f"{h}:{m}")

0:01 | sai nếu không đệm: 0:1
10:02 | sai nếu không đệm: 10:2
4:51 | sai nếu không đệm: 4:51
11:59 | sai nếu không đệm: 11:59


### 2.3. Toán tử bit `&`, `>>`, `<<`

- `x >> k`: dịch phải `k` bit (≈ chia `2^k` lấy nguyên). `x & 1`: lấy bit thấp nhất.
- `1 << k`: số chỉ có bit thứ `k` bằng 1 (= `2^k`).
- `mask & 0b111111`: giữ lại 6 bit thấp.

Dùng để tách một số 10 bit thành 4 bit giờ (cao) + 6 bit phút (thấp).

In [4]:
mask = 0b0100110011        # 4 bit giờ '0100' ghép 6 bit phút '110011'
print("mask =", mask)
print("giờ  = mask >> 6       =", mask >> 6)
print("phút = mask & 0b111111 =", mask & 0b111111, "(0b111111 =", 0b111111, ")")
print("1 << 5 =", 1 << 5, "| (51 >> 4) & 1 =", (51 >> 4) & 1, "(bit thứ 4 của 51)")

mask = 307
giờ  = mask >> 6       = 4
phút = mask & 0b111111 = 51 (0b111111 = 63 )
1 << 5 = 32 | (51 >> 4) & 1 = 1 (bit thứ 4 của 51)


## 3. Cách nghĩ trực giác — "chọn đèn nào sáng"

Đọc đề, người mới thường nghĩ: "có 10 đèn, chọn `turnedOn` đèn để bật → tính giờ, phút → bỏ những tổ hợp vượt 11 giờ hoặc 59 phút".

Số cách chọn là tổ hợp `C(10, k)` — tối đa `C(10, 5) = 252`. Hoàn toàn khả thi! Cách này dẫn tới **quay lui** (mục 6). Nhưng trước tiên có một góc nhìn **đơn giản hơn nhiều**.

In [5]:
from math import comb
print([comb(10, k) for k in range(11)], "| tổng =", sum(comb(10, k) for k in range(11)), "= 2^10")

[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1] | tổng = 1024 = 2^10


### 3.1. Insight — đảo chiều: liệt kê **đáp án** thay vì liệt kê **đèn**

Chỉ có `12 × 60 = 720` thời điểm hợp lệ. Với mỗi `(h, m)`, số đèn sáng **chính là** `popcount(h) + popcount(m)`. Vậy:

> Duyệt mọi `h ∈ [0, 11]`, `m ∈ [0, 59]`; giữ lại những cặp có `popcount(h) + popcount(m) == turnedOn`.

Ưu điểm: **không bao giờ** sinh thời điểm không hợp lệ, không cần kiểm tra `h ≤ 11`, `m ≤ 59`, và code chỉ vài dòng. Đây cũng là gợi ý của LeetCode: "so sánh số bit 1" và "tính trước mọi thời điểm".

## 4. Cách 1 — Duyệt mọi thời điểm (lời giải chính)

**Đoạn 1:** hai vòng lặp `for h in range(12)` và `for m in range(60)`.

**Đoạn 2:** đếm đèn `bin(h).count('1') + bin(m).count('1')`.

**Đoạn 3:** nếu bằng `turnedOn` → thêm chuỗi `f"{h}:{m:02d}"`.

In [6]:
def read_watch_enumerate(turnedOn: int) -> list[str]:
    result = []
    for h in range(12):
        for m in range(60):
            if bin(h).count("1") + bin(m).count("1") == turnedOn:
                result.append(f"{h}:{m:02d}")
    return result

print(read_watch_enumerate(1))
print(read_watch_enumerate(9))
print("turnedOn=2 có", len(read_watch_enumerate(2)), "thời điểm, ví dụ:", read_watch_enumerate(2)[:6])

['0:01', '0:02', '0:04', '0:08', '0:16', '0:32', '1:00', '2:00', '4:00', '8:00']
[]
turnedOn=2 có 44 thời điểm, ví dụ: ['0:03', '0:05', '0:06', '0:09', '0:10', '0:12']


## 5. TRACE — `turnedOn = 1` (chỉ in các cặp được chọn và vài cặp bị loại tiêu biểu)

| `h` | `m` | bit(h) | bit(m) | Tổng | Chọn? |
|---:|---:|---:|---:|---:|:-:|
| 0 | 0 | 0 | 0 | 0 | ✘ |
| 0 | 1 | 0 | 1 | 1 | ✔ `"0:01"` |
| 0 | 3 | 0 | 2 | 2 | ✘ |
| 0 | 32 | 0 | 1 | 1 | ✔ `"0:32"` |
| 1 | 0 | 1 | 0 | 1 | ✔ `"1:00"` |
| 3 | 0 | 2 | 0 | 2 | ✘ |
| 8 | 0 | 1 | 0 | 1 | ✔ `"8:00"` |

Với 1 đèn: hoặc đèn sáng ở hàng giờ (giờ là lũy thừa của 2: 1, 2, 4, 8) và phút = 0; hoặc ở hàng phút (1, 2, 4, 8, 16, 32) và giờ = 0 → tổng **4 + 6 = 10** thời điểm.

In [7]:
def trace_watch(turnedOn: int, show_rejected=(0, 3)) -> list[str]:
    result = []
    for h in range(12):
        for m in range(60):
            bh, bm = bin(h).count("1"), bin(m).count("1")
            if bh + bm == turnedOn:
                result.append(f"{h}:{m:02d}")
                print(f"h={h:>2} ({h:04b}) m={m:>2} ({m:06b}) -> {bh}+{bm}={bh + bm} ✔ chọn {result[-1]}")
            elif h == 0 and m in show_rejected:
                print(f"h={h:>2} ({h:04b}) m={m:>2} ({m:06b}) -> {bh}+{bm}={bh + bm} ✘ loại")
    print("Tổng số thời điểm:", len(result))
    return result

_ = trace_watch(1)

h= 0 (0000) m= 0 (000000) -> 0+0=0 ✘ loại
h= 0 (0000) m= 1 (000001) -> 0+1=1 ✔ chọn 0:01
h= 0 (0000) m= 2 (000010) -> 0+1=1 ✔ chọn 0:02
h= 0 (0000) m= 3 (000011) -> 0+2=2 ✘ loại
h= 0 (0000) m= 4 (000100) -> 0+1=1 ✔ chọn 0:04
h= 0 (0000) m= 8 (001000) -> 0+1=1 ✔ chọn 0:08
h= 0 (0000) m=16 (010000) -> 0+1=1 ✔ chọn 0:16
h= 0 (0000) m=32 (100000) -> 0+1=1 ✔ chọn 0:32
h= 1 (0001) m= 0 (000000) -> 1+0=1 ✔ chọn 1:00
h= 2 (0010) m= 0 (000000) -> 1+0=1 ✔ chọn 2:00
h= 4 (0100) m= 0 (000000) -> 1+0=1 ✔ chọn 4:00
h= 8 (1000) m= 0 (000000) -> 1+0=1 ✔ chọn 8:00
Tổng số thời điểm: 10


`f"{h:04b}"`: in `h` dạng **nhị phân** (`b`), độ rộng 4, đệm 0 → giống hệt hàng đèn giờ. `{m:06b}` tương tự cho 6 đèn phút.

## 6. Cách 2 — Quay lui (backtracking): chọn đèn nào bật

Mô hình hóa 10 đèn thành list giá trị: 4 đèn giờ `[8, 4, 2, 1]` và 6 đèn phút `[32, 16, 8, 4, 2, 1]`.

Hàm đệ quy `backtrack(start, remaining, h, m)`:
- `remaining == 0` → đã bật đủ đèn → ghi nhận `(h, m)`.
- Ngược lại, thử bật đèn `i` từ `start` tới 9 (chỉ đi **về phía trước** để không đếm trùng tổ hợp):
  - nếu là đèn giờ, cộng vào `h`; nếu `h > 11` thì **cắt nhánh (prune)**;
  - nếu là đèn phút, cộng vào `m`; nếu `m > 59` thì cắt nhánh;
  - gọi `backtrack(i + 1, remaining - 1, ...)`.

In [8]:
LEDS = [(8, 0), (4, 0), (2, 0), (1, 0), (0, 32), (0, 16), (0, 8), (0, 4), (0, 2), (0, 1)]
# mỗi đèn là (giá trị cộng vào giờ, giá trị cộng vào phút)

def read_watch_backtrack(turnedOn: int) -> list[str]:
    result = []

    def backtrack(start: int, remaining: int, h: int, m: int) -> None:
        if remaining == 0:
            result.append(f"{h}:{m:02d}")
            return
        for i in range(start, len(LEDS)):
            dh, dm = LEDS[i]
            nh, nm = h + dh, m + dm
            if nh > 11 or nm > 59:
                continue                       # cắt nhánh: thời điểm không hợp lệ
            backtrack(i + 1, remaining - 1, nh, nm)

    backtrack(0, turnedOn, 0, 0)
    return result

print(sorted(read_watch_backtrack(1)))
print(read_watch_backtrack(9))
print(sorted(read_watch_backtrack(2)) == sorted(read_watch_enumerate(2)))

['0:01', '0:02', '0:04', '0:08', '0:16', '0:32', '1:00', '2:00', '4:00', '8:00']
[]
True


**Hàm lồng (nested function):** `backtrack` định nghĩa **bên trong** `read_watch_backtrack` nên đọc/ghi được list `result` của hàm ngoài (gọi là **closure**). `result.append` sửa list tại chỗ nên không cần `nonlocal`.

**Vì sao `range(start, ...)` và `i + 1`?** Để mỗi tổ hợp đèn chỉ sinh **một lần** theo thứ tự chỉ số tăng dần (tránh sinh `{đèn 1, đèn 5}` và `{đèn 5, đèn 1}` như hai đáp án).

In [9]:
def trace_backtrack(turnedOn: int, limit: int = 14) -> None:
    lines = []

    def backtrack(start, remaining, h, m, depth):
        if len(lines) >= limit:
            return
        indent = "  " * depth
        if remaining == 0:
            lines.append(f"{indent}✔ ghi nhận {h}:{m:02d}")
            return
        for i in range(start, len(LEDS)):
            dh, dm = LEDS[i]
            nh, nm = h + dh, m + dm
            if len(lines) >= limit:
                return
            if nh > 11 or nm > 59:
                lines.append(f"{indent}bật đèn {i} -> h={nh}, m={nm}: ✘ cắt nhánh")
                continue
            lines.append(f"{indent}bật đèn {i} -> h={nh}, m={nm}")
            backtrack(i + 1, remaining - 1, nh, nm, depth + 1)

    backtrack(0, turnedOn, 0, 0, 0)
    print("\n".join(lines))
    print("... (chỉ in", limit, "dòng đầu)")

trace_backtrack(2)

bật đèn 0 -> h=8, m=0
  bật đèn 1 -> h=12, m=0: ✘ cắt nhánh
  bật đèn 2 -> h=10, m=0
    ✔ ghi nhận 10:00
  bật đèn 3 -> h=9, m=0
    ✔ ghi nhận 9:00
  bật đèn 4 -> h=8, m=32
    ✔ ghi nhận 8:32
  bật đèn 5 -> h=8, m=16
    ✔ ghi nhận 8:16
  bật đèn 6 -> h=8, m=8
    ✔ ghi nhận 8:08
  bật đèn 7 -> h=8, m=4
    ✔ ghi nhận 8:04
... (chỉ in 14 dòng đầu)


Đọc trace: bật đèn 0 (8 giờ) rồi đèn 1 (thêm 4 giờ → 12) → **cắt nhánh** vì 12 > 11; tiếp tục thử đèn 2 (8 + 2 = 10 giờ) → hợp lệ → ghi nhận `10:00`...

## 7. Cách 3 — Duyệt mọi mặt nạ 10 bit (bitmask)

Mỗi cấu hình 10 đèn là một số `mask` từ `0` đến `2^10 − 1 = 1023`. Giờ = `mask >> 6` (4 bit cao), phút = `mask & 63` (6 bit thấp). Giữ lại mask có `popcount == turnedOn`, `h ≤ 11`, `m ≤ 59`.

In [10]:
def read_watch_bitmask(turnedOn: int) -> list[str]:
    result = []
    for mask in range(1 << 10):           # 0..1023
        if mask.bit_count() != turnedOn:
            continue
        h, m = mask >> 6, mask & 0b111111
        if h <= 11 and m <= 59:
            result.append(f"{h}:{m:02d}")
    return result

print(sorted(read_watch_bitmask(1)))
print(sorted(read_watch_bitmask(2)) == sorted(read_watch_enumerate(2)))

['0:01', '0:02', '0:04', '0:08', '0:16', '0:32', '1:00', '2:00', '4:00', '8:00']
True


## 8. Cách 4 — Tính trước bảng tra (precompute)

Nếu hàm được gọi **rất nhiều lần**, tính sẵn một lần: `TABLE[k]` = list thời điểm có đúng `k` đèn. Mỗi lần gọi chỉ trả `TABLE[turnedOn]`. Đây là gợi ý thứ hai của đề ("precomputing all possible times").

In [11]:
TABLE = [[] for _ in range(11)]
for h in range(12):
    for m in range(60):
        TABLE[h.bit_count() + m.bit_count()].append(f"{h}:{m:02d}")

def read_watch_table(turnedOn: int) -> list[str]:
    return list(TABLE[turnedOn])          # trả bản sao để người gọi sửa không ảnh hưởng bảng

print([len(row) for row in TABLE], "| tổng =", sum(len(row) for row in TABLE))
print(read_watch_table(1))

[1, 10, 44, 112, 181, 190, 126, 48, 8, 0, 0] | tổng = 720
['0:01', '0:02', '0:04', '0:08', '0:16', '0:32', '1:00', '2:00', '4:00', '8:00']


Dòng in số lượng theo `k`: `[1, 10, 44, 112, 181, 190, 126, 48, 8, 0, 0]`, tổng **720** — đúng bằng số thời điểm. Ta cũng thấy rõ `k = 9, 10` cho 0 kết quả, `k = 8` chỉ có 8 thời điểm (giờ 7 hoặc 11 — 3 bit; phút 31, 47, 55, 59 — 5 bit).

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def readBinaryWatch(self, turnedOn: int) -> list[str]:
        result = []
        for h in range(12):
            for m in range(60):
                if bin(h).count("1") + bin(m).count("1") == turnedOn:
                    result.append(f"{h}:{m:02d}")
        return result


sol = Solution()
print(sol.readBinaryWatch(1))
print(sol.readBinaryWatch(9))

['0:01', '0:02', '0:04', '0:08', '0:16', '0:32', '1:00', '2:00', '4:00', '8:00']
[]


Bản ngắn (list comprehension 2 vòng `for`):

```python
class Solution:
    def readBinaryWatch(self, turnedOn: int) -> list[str]:
        return [f"{h}:{m:02d}" for h in range(12) for m in range(60)
                if (h.bit_count() + m.bit_count()) == turnedOn]
```

(Trong list comprehension, nhiều `for` lồng nhau đọc **từ trái sang phải** như vòng lặp ngoài → trong.)

## 10. Vì sao đúng? + Độ phức tạp

**Đúng (Cách 1):** một thời điểm hiển thị được ⇔ `0 ≤ h ≤ 11`, `0 ≤ m ≤ 59` và số đèn sáng bằng `turnedOn`. Số đèn sáng ở hàng giờ đúng bằng số bit 1 của `h` (mỗi đèn là một bit), tương tự với phút. Hai vòng lặp duyệt **đúng và đủ** mọi `(h, m)` hợp lệ, mỗi cặp một lần, nên kết quả **không thiếu, không thừa, không trùng**.

**Đúng (Cách 2):** mỗi tập đèn được sinh đúng một lần (chỉ số tăng dần); nhánh bị cắt chỉ chứa tập đèn có `h > 11` hoặc `m > 59` — vì giá trị chỉ **tăng** khi bật thêm đèn, mọi mở rộng của nhánh đó cũng không hợp lệ, nên cắt an toàn.

| Cách | Time | Space (không tính output) |
|---|---|---|
| 1. Duyệt 720 thời điểm | `O(12·60)` = `O(1)` | `O(1)` |
| 2. Quay lui | `O(C(10, k))` ≤ 252 lá | `O(10)` ngăn xếp đệ quy |
| 3. Bitmask | `O(2^10)` = `O(1)` | `O(1)` |
| 4. Bảng tra | tiền xử lý `O(720)`, mỗi lần gọi `O(số kết quả)` | `O(720)` |

Vì input nhỏ cố định, mọi cách đều là hằng số; điều quan trọng là **cách nghĩ** và độ gọn của code.

## 11. Bộ test: mẫu đề, biên, đối chiếu (so sánh sau khi sắp xếp vì thứ tự tùy ý)

In [13]:
methods = {
    "enumerate": read_watch_enumerate,
    "backtrack": read_watch_backtrack,
    "bitmask": read_watch_bitmask,
    "table": read_watch_table,
    "Solution": Solution().readBinaryWatch,
}
expected1 = ["0:01", "0:02", "0:04", "0:08", "0:16", "0:32", "1:00", "2:00", "4:00", "8:00"]
assert sorted(Solution().readBinaryWatch(1)) == sorted(expected1)
assert Solution().readBinaryWatch(9) == [] and Solution().readBinaryWatch(10) == []
assert Solution().readBinaryWatch(0) == ["0:00"]
assert sorted(Solution().readBinaryWatch(8)) == sorted(["7:31", "7:47", "7:55", "7:59", "11:31", "11:47", "11:55", "11:59"])
assert [len(row) for row in TABLE] == [1, 10, 44, 112, 181, 190, 126, 48, 8, 0, 0]
for k in range(11):
    ref = sorted(read_watch_enumerate(k))
    for name, f in methods.items():
        got = f(k)
        assert sorted(got) == ref, (name, k)
        assert len(got) == len(set(got)), (name, "có phần tử trùng")
    print(f"turnedOn={k:>2}: {len(ref):>3} thời điểm — mọi cách khớp")

turnedOn= 0:   1 thời điểm — mọi cách khớp
turnedOn= 1:  10 thời điểm — mọi cách khớp
turnedOn= 2:  44 thời điểm — mọi cách khớp
turnedOn= 3: 112 thời điểm — mọi cách khớp
turnedOn= 4: 181 thời điểm — mọi cách khớp
turnedOn= 5: 190 thời điểm — mọi cách khớp
turnedOn= 6: 126 thời điểm — mọi cách khớp
turnedOn= 7:  48 thời điểm — mọi cách khớp
turnedOn= 8:   8 thời điểm — mọi cách khớp
turnedOn= 9:   0 thời điểm — mọi cách khớp
turnedOn=10:   0 thời điểm — mọi cách khớp


In [14]:
# Kiểm tra định dạng của MỌI chuỗi kết quả
import re
pattern = re.compile(r"^(?:[0-9]|1[01]):[0-5][0-9]$")     # giờ 0..11 không có 0 đầu, phút 00..59
for k in range(11):
    for s in Solution().readBinaryWatch(k):
        assert pattern.match(s), s
print("TABLE[8] =", sorted(TABLE[8]))
print("Mọi chuỗi đúng định dạng 'H:MM'")

TABLE[8] = ['11:31', '11:47', '11:55', '11:59', '7:31', '7:47', '7:55', '7:59']
Mọi chuỗi đúng định dạng 'H:MM'


Cell trên dùng **biểu thức chính quy** (`re`, thư viện chuẩn) chỉ để kiểm thử: `[0-9]|1[01]` là giờ `0..9` hoặc `10, 11`; `[0-5][0-9]` là phút `00..59`; `^...$` bắt buộc khớp toàn chuỗi.

## 12. Lỗi người mới hay gặp

### Lỗi 1 — Phút không đệm 0: `f"{h}:{m}"` → `"10:2"`.
### Lỗi 2 — Giờ có đệm 0: `f"{h:02d}:{m:02d}"` → `"01:00"`.
### Lỗi 3 — Duyệt giờ tới 12 hoặc phút tới 60 (`range(13)`, `range(61)`) → sinh `"12:00"`, `"0:60"`.
### Lỗi 4 — Quay lui không dùng `start` → sinh trùng tổ hợp (cùng một thời điểm xuất hiện nhiều lần).

In [15]:
def wrong_no_pad(k):
    return [f"{h}:{m}" for h in range(12) for m in range(60) if h.bit_count() + m.bit_count() == k]

def wrong_pad_hour(k):
    return [f"{h:02d}:{m:02d}" for h in range(12) for m in range(60) if h.bit_count() + m.bit_count() == k]

def wrong_range(k):
    return [f"{h}:{m:02d}" for h in range(13) for m in range(61) if h.bit_count() + m.bit_count() == k]

def wrong_backtrack_no_start(k):
    result = []
    def bt(remaining, h, m, used):
        if remaining == 0:
            result.append(f"{h}:{m:02d}")
            return
        for i in range(len(LEDS)):               # luôn từ 0 -> trùng
            if i in used:
                continue
            nh, nm = h + LEDS[i][0], m + LEDS[i][1]
            if nh <= 11 and nm <= 59:
                bt(remaining - 1, nh, nm, used | {i})
    bt(k, 0, 0, frozenset())
    return result

print("Lỗi 1:", [s for s in wrong_no_pad(1) if s.startswith("0:")][:3])
print("Lỗi 2:", wrong_pad_hour(1)[-3:])
print("Lỗi 3: thừa (k=2)", sorted(set(wrong_range(2)) - set(read_watch_enumerate(2))), "| thừa (k=4) ví dụ", sorted(set(wrong_range(4)) - set(read_watch_enumerate(4)))[:4])
r = wrong_backtrack_no_start(2)
print(f"Lỗi 4: {len(r)} kết quả nhưng chỉ {len(set(r))} khác nhau")

Lỗi 1: ['0:1', '0:2', '0:4']
Lỗi 2: ['02:00', '04:00', '08:00']
Lỗi 3: thừa (k=2) ['12:00'] | thừa (k=4) ví dụ ['0:60', '12:03', '12:05', '12:06']
Lỗi 4: 88 kết quả nhưng chỉ 44 khác nhau


(`frozenset` là set **bất biến**; `used | {i}` tạo set mới có thêm `i` — để mỗi nhánh đệ quy giữ bản riêng.)

## 13. Mẹo nhận dạng pattern

- **Không gian đáp án nhỏ** (ở đây 720) → **liệt kê đáp án và lọc** thường gọn hơn sinh cấu hình rồi kiểm tra.
- "Số đèn sáng / số phần tử được chọn" với các giá trị lũy thừa của 2 → **popcount**.
- "Chọn `k` trong `n` món" với ràng buộc tổng → **quay lui + cắt nhánh**, hoặc **duyệt bitmask** khi `n ≤ ~20`.
- Định dạng đầu ra là một phần của đề — đọc kỹ quy tắc 0 đứng đầu.
- Bài anh em: **191 Number of 1 Bits**, **338 Counting Bits**, **77 Combinations**, **78 Subsets** (bitmask/backtracking), **1239**.

## 14. Biến thể

### 14.1. Đồng hồ 24 giờ (5 đèn giờ, 0–23)

Chỉ đổi `range(12)` thành `range(24)` — lợi thế của cách liệt kê đáp án.

In [16]:
def read_watch_24h(turnedOn: int) -> list[str]:
    return [f"{h}:{m:02d}" for h in range(24) for m in range(60) if h.bit_count() + m.bit_count() == turnedOn]

print(len(read_watch_24h(1)), read_watch_24h(1)[-3:])     # thêm "16:00"

11 ['4:00', '8:00', '16:00']


### 14.2. Ngược lại: cho chuỗi thời gian, cần bao nhiêu đèn?

In [17]:
def leds_needed(time_str: str) -> int:
    h, m = map(int, time_str.split(":"))       # split tách theo ':', map(int, ...) đổi từng phần sang số
    return h.bit_count() + m.bit_count()

print(leds_needed("4:51"), leds_needed("11:59"), leds_needed("0:00"))

5 8 0


## 15. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Viết dạng 4 bit giờ + 6 bit phút của `"7:31"` và `"11:59"`. Mỗi cái cần bao nhiêu đèn?

**Bài 2 (dự đoán output).** `f"{5:06b}"`, `f"{7:02d}"`, `(45 >> 2) & 1` bằng gì?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_enumerate` (Cách 1).

**Bài 4 (viết lại).** Viết lại Cách 3 (bitmask) **không** dùng `bit_count` — tự đếm bit bằng vòng `while x: x &= x - 1` (mẹo Brian Kernighan: mỗi lần xóa bit 1 thấp nhất).

**Bài 5 (biến thể).** `count_times(turnedOn)` trả về **số lượng** thời điểm (không cần list), dùng phép nhân: đếm số giờ có `a` bit 1 và số phút có `b` bit 1, cộng `hours[a] * minutes[b]` với `a + b = turnedOn`.

**Bài 6 (mini challenge).** Đồng hồ có thêm hàng **giây** 6 đèn (0–59). Trả về mọi `"H:MM:SS"` với tổng `turnedOn` đèn. Chỉ cần trả **số lượng** cho `turnedOn = 3` để kiểm tra.

In [18]:
def exercise_enumerate(turnedOn: int) -> list[str]:
    # TODO 1: result = []
    # TODO 2: for h in range(12): for m in range(60):
    # TODO 3: nếu số bit 1 của h + của m == turnedOn: append f"{h}:{m:02d}"
    pass

# assert sorted(exercise_enumerate(1)) == sorted(read_watch_enumerate(1))

In [19]:
def popcount_kernighan(x: int) -> int:
    # TODO
    pass

def exercise_bitmask(turnedOn: int) -> list[str]:
    # TODO: dùng popcount_kernighan
    pass

# assert popcount_kernighan(51) == 4
# assert sorted(exercise_bitmask(3)) == sorted(read_watch_enumerate(3))

In [20]:
def count_times(turnedOn: int) -> int:
    # TODO
    pass

# assert [count_times(k) for k in range(11)] == [len(row) for row in TABLE]

In [21]:
def read_watch_with_seconds_count(turnedOn: int) -> int:
    # TODO
    pass

# assert read_watch_with_seconds_count(0) == 1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đổi từng số sang nhị phân | 7 = `0111`, 31 = `011111` | đếm số chữ `1` |
| 2 | `b` = nhị phân, `d` = thập phân | `06` đệm 0 tới 6 ký tự | 45 = `101101`, dịch 2 → `1011` |
| 3 | xem mục 4 | `bin(x).count('1')` | f-string `{m:02d}` |
| 4 | `x & (x - 1)` xóa bit 1 thấp nhất | đếm số lần tới khi `x == 0` | thay `mask.bit_count()` bằng hàm của bạn |
| 5 | phân loại giờ theo popcount | `hours = [0]*5`, `minutes = [0]*7` | tổng các tích với `a + b == k` |
| 6 | thêm vòng `for s in range(60)` | hoặc dùng mảng đếm như bài 5 | ba mảng đếm, ba vòng nhỏ |

## 17. Đáp án tham khảo

**Bài 1:** `"7:31"` → giờ `0111`, phút `011111` → 3 + 5 = **8** đèn. `"11:59"` → `1011`, `111011` → 3 + 5 = **8** đèn (cả hai đều nằm trong `TABLE[8]`).

**Bài 2:** `"000101"`, `"07"`, `1` (45 = `101101`, `>> 2` = `1011` = 11, bit thấp nhất là 1).

In [22]:
def answer_enumerate(turnedOn):
    result = []
    for h in range(12):
        for m in range(60):
            if bin(h).count("1") + bin(m).count("1") == turnedOn:
                result.append(f"{h}:{m:02d}")
    return result

def answer_popcount_kernighan(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count

def answer_bitmask(turnedOn):
    out = []
    for mask in range(1 << 10):
        if answer_popcount_kernighan(mask) == turnedOn:
            h, m = mask >> 6, mask & 63
            if h < 12 and m < 60:
                out.append(f"{h}:{m:02d}")
    return out

def answer_count_times(turnedOn):
    hours = [0] * 5
    minutes = [0] * 7
    for h in range(12):
        hours[h.bit_count()] += 1
    for m in range(60):
        minutes[m.bit_count()] += 1
    return sum(hours[a] * minutes[turnedOn - a] for a in range(5) if 0 <= turnedOn - a <= 6)

def answer_with_seconds_count(turnedOn):
    hours, minutes = [0] * 5, [0] * 7
    for h in range(12):
        hours[h.bit_count()] += 1
    for m in range(60):
        minutes[m.bit_count()] += 1
    total = 0
    for a in range(5):
        for b_ in range(7):
            c = turnedOn - a - b_
            if 0 <= c <= 6:
                total += hours[a] * minutes[b_] * minutes[c]   # giây dùng cùng bảng với phút
    return total

print("Bài 1:", leds_needed("7:31"), leds_needed("11:59"), f"| 7={7:04b}, 31={31:06b}, 11={11:04b}, 59={59:06b}")
print("Bài 2:", f"{5:06b}", f"{7:02d}", (45 >> 2) & 1)
assert sorted(answer_enumerate(1)) == sorted(read_watch_enumerate(1))
assert answer_popcount_kernighan(51) == 4
assert sorted(answer_bitmask(3)) == sorted(read_watch_enumerate(3))
assert [answer_count_times(k) for k in range(11)] == [len(row) for row in TABLE]
assert answer_with_seconds_count(0) == 1
brute3 = sum(1 for h in range(12) for m in range(60) for s in range(60)
             if h.bit_count() + m.bit_count() + s.bit_count() == 3)
assert answer_with_seconds_count(3) == brute3
print("Bài 6: số thời điểm H:MM:SS với 3 đèn =", brute3)
print("Đáp án PASS toàn bộ!")

Bài 1: 8 8 | 7=0111, 31=011111, 11=1011, 59=111011
Bài 2: 000101 07 1
Bài 6: số thời điểm H:MM:SS với 3 đèn = 546
Đáp án PASS toàn bộ!


## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def readBinaryWatch(self, turnedOn: int) -> list[str]:
        return [f"{h}:{m:02d}"                      # giờ không đệm, phút 2 chữ số
                for h in range(12)                  # 0..11
                for m in range(60)                  # 0..59
                if bin(h).count("1") + bin(m).count("1") == turnedOn]
```

**Checklist**
- [ ] Giờ `0..11` (`range(12)`), phút `0..59` (`range(60)`)?
- [ ] Phút luôn 2 chữ số (`:02d`), giờ **không** đệm 0?
- [ ] `turnedOn = 0` → `["0:00"]`; `turnedOn ≥ 9` → `[]`?
- [ ] Nếu quay lui: dùng `start` để không trùng, cắt nhánh khi `h > 11` / `m > 59`?
- [ ] Khi tự test: so sánh `sorted(...)` vì thứ tự tùy ý?